In [1]:
## Hugging Face Transformers local LLM -> TinyLlama/TinyLlama-1.1B-Chat-v1.0
import os

BASE_PATH = r"D:\\CODE\\Alzheimer_Detection_And_Classification\\rag"
PDF_PATH = os.path.join(BASE_PATH, "data", "Treatment_doc")
VECTOR_PATH = os.path.join(BASE_PATH, "vector_db")

EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
COLLECTION_NAME = 'alzheimer_treatment_docs'
TOP_K = 4


In [2]:
# !pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers requests transformers accelerate torch


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=VECTOR_PATH,
)

print('Connected to vector DB:', VECTOR_PATH)


C:\Users\arnab\anaconda3\envs\p_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4925.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connected to vector DB: D:\\CODE\\Alzheimer_Detection_And_Classification\\rag\vector_db


In [3]:
def retrieve_context(stage, query, top_k=4):
    combined_query = f'Alzheimer stage: {stage}. Question: {query}'
    results = vector_store.similarity_search_with_score(combined_query, k=top_k)
    return results


def build_augmented_context(results):
    context_blocks = []
    sources = []

    for doc, score in results:
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', 'N/A')
        chunk_id = doc.metadata.get('chunk_id', 'N/A')
        text = doc.page_content.strip()

        context_blocks.append(
            f'Source: {source} | Page: {page} | Chunk: {chunk_id} | Score: {score}\n{text}'
        )
        sources.append({
            'source': source,
            'page': page,
            'chunk_id': chunk_id,
            'score': score,
            'content': text[:500]
        })

    return '\n\n'.join(context_blocks), sources


In [4]:
SYSTEM_PROMPT = '''
You are a professional Alzheimer treatment assistant for document-grounded clinical education support.
Answer ONLY from the provided retrieved context.
Do not use outside knowledge.
If the context is not enough, say clearly that the documents do not contain enough information.

Use this format:
1. Current stage summary
2. What to do now
3. Medicinal treatment mentioned in the documents
4. Non-medicinal treatment mentioned in the documents
5. Red flags / when to seek specialist help
6. Source-based note
'''

def build_user_prompt(stage, query, context):
    return f'''
Alzheimer stage: {stage}
User question: {query}

Retrieved context:
{context}

Generate the final answer using only the retrieved context.
'''


In [6]:
import torch
from transformers import pipeline

HF_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

generator = pipeline(
    'text-generation',
    model=HF_MODEL,
    torch_dtype=dtype,
    device=device
)

print('Loaded Hugging Face model:', HF_MODEL)
print('Using CUDA:', torch.cuda.is_available())


C:\Users\arnab\anaconda3\envs\p_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arnab\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████

Loaded Hugging Face model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Using CUDA: False


In [7]:
def generate_with_transformers(system_prompt, user_prompt, max_new_tokens=700):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ]

    output = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.2,
        return_full_text=False
    )

    if isinstance(output, list) and len(output) > 0:
        item = output[0]
        if 'generated_text' in item:
            generated = item['generated_text']
            if isinstance(generated, list) and len(generated) > 0:
                last_msg = generated[-1]
                if isinstance(last_msg, dict) and 'content' in last_msg:
                    return last_msg['content'].strip()
            return str(generated).strip()
    return str(output).strip()


In [9]:
stage = 'early'
query = 'What should be done now? Give medicinal treatment and non-medicinal treatment.'

results = retrieve_context(stage, query, top_k=TOP_K)
augmented_context, sources = build_augmented_context(results)
user_prompt = build_user_prompt(stage, query, augmented_context)

hf_answer = generate_with_transformers(SYSTEM_PROMPT, user_prompt)
print(hf_answer)


Both `max_new_tokens` (=700) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


User question: "What should be done now? Give medicinal treatment and non-medicinal treatment."

Retrieved context:
Source: D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\the_dementia_guide.pdf | Page: 35 | Chunk: chunk-1157 | Score: 0.6172906756401062
The dementia guide
34
There is no known cure for dementia yet. Treatment includes drug and non-drug approaches, looking after other medical conditions and making changes to your lifestyle. With a combination of these, it is possible to live well with dementia for many years. Four drugs have been developed to treat Alzheimer’s disease:

1. Donepezil
2. Rivastigmine
3. Galantamine
4. Memantine

These drugs may reduce the symptoms of Alzheimer’s disease or stop them getting worse for a while. You may also be given one of these if you have dementia with Lewy bodies, Parkinson’s disease dementia, or mixed dementia.

Key points: Treatments

1. Caring for a person with Alzheimer’s disease: Your easy-to-use guide
When this

In [ ]:
response_hf = ask_rag(
    stage='mild',
    query='What should be done now? Mention medicinal treatment and non-medicinal treatment.',
    top_k=4,
    method='hf'
)

print('HF ANSWER:\n')
print(response_hf['answer'])


In [10]:
print(augmented_context)

Source: D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\the_dementia_guide.pdf | Page: 35 | Chunk: chunk-1157 | Score: 0.6172906756401062
The dementia guide
34
There is no known cure for dementia yet. Treatment 
includes drug and non-drug approaches, looking 
after other medical conditions and making changes 
to your lifestyle. With a combination of these, it is 
possible to live well with dementia for many years.
Four drugs have been developed to treat 
Alzheimer’s disease:
■  donepezil
■  rivastigmine
■  galantamine
■  memantine.
 
These drugs may reduce the symptoms of 
Alzheimer’s disease or stop them getting worse  
for a while. You may also be given one of these if  
you have dementia with Lewy bodies, Parkinson’s 
disease dementia or mixed dementia.
Key points: Treatments

Source: D:\CODE\Alzheimer_Detection_And_Classification\rag\data\Treatment_doc\A Guide to Caring for a Person with Alzheimer's Disease (1).pdf | Page: 75 | Chunk: chunk-158 | Score: 0.6318